In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pyarrow.parquet as pq
import pandas as pd 
from pathlib import Path
from PIL import Image
import torch
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

In [19]:
PARQUET_PATH = Path("/Users/mainframe/Workspace/Graduate/notebooks/plant_gotchi_train.parquet")
PHOTO_PATH = Path("/Users/mainframe/Workspace/Graduate/dataset/train/photos")

df = pq.read_table(source=PARQUET_PATH, use_threads=True).to_pandas()

scaler = StandardScaler()
cols_to_scale = ['temp', 'humid', 'co2', 'light', 'stem_area', 'leaf_area']
df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

In [20]:
class PlantaPipeline:
    def __init__(self, size=512):
        # This is our 'Transformation Contract'
        self.transform = transforms.Compose([
            transforms.Resize((size, size)),
            transforms.ToTensor(),
            # Standard ImageNet normalization (works well for most lettuce)
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                                 std=[0.229, 0.224, 0.225])
        ])

    def process(self, image_path: Path):
        """Reads and transforms a single image for the M4 Max NPU."""
        try:
            # 1. Open the image
            img = Image.open(image_path).convert('RGB')
            
            # 2. Apply the transformations
            img_tensor = self.transform(img)
            
            # 3. Add a batch dimension (C, H, W) -> (1, C, H, W)
            # Neural networks expect a 'batch' even if it's just one image
            return img_tensor.unsqueeze(0)
            
        except Exception as e:
            print(f"Failed to process {image_path.name}: {e}")
            return None

In [21]:
class PlantaDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.df = dataframe
        self.img_dir = Path(img_dir)
        self.transform = transform
        
        # Prepare tabular features for the MLP branch
        # We normalize these so the model doesn't choke on large numbers
        self.tabular_cols = ['temp', 'humid', 'co2', 'light', 'stem_area', 'leaf_area']
        
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # 1. Get the row data
        row = self.df.iloc[idx]
        
        # 2. Handle the Image
        # Assuming your file_name needs an extension like .jpg
        img_path = self.img_dir / f"{row['file_name']}.jpg"
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
            
        # 3. Handle the Tabular Data (MLP Input)
        # Convert the row features into a FloatTensor
        tab_data = torch.tensor(row[self.tabular_cols].values.astype('float32'))
        
        # 4. Handle the Label (Target)
        # Convert 'growth' (1, 2, 3) to a float for Regression
        label = torch.tensor([float(row['growth'])])
        
        return image, tab_data, label

In [22]:
train_ds = PlantaDataset(
    dataframe=df, 
    img_dir=PHOTO_PATH, 
    transform=pipeline.transform # Using the pipeline we built earlier
)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=0)

In [23]:
class PlantaGotchiMultiModal(nn.Module):
    def __init__(self, num_tabular_features=6):
        super(PlantaGotchiMultiModal, self).__init__()
        
        # 1. Vision Branch
        self.vision_branch = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # 256
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # 128
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten() # Result: 64 features
        )
        
        # 2. Tabular Branch
        self.tabular_branch = nn.Sequential(
            nn.Linear(num_tabular_features, 16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.ReLU()
        )
        
        # 3. The Fusion Head (The 'Decision')
        self.regressor = nn.Sequential(
            nn.Linear(64 + 16, 32),
            nn.ReLU(),
            nn.Linear(32, 1) # Output: The Age/Stage
        )

    def forward(self, img, tab):
        v_feat = self.vision_branch(img)
        t_feat = self.tabular_branch(tab)
        
        # 'Glue' the vectors together
        combined = torch.cat((v_feat, t_feat), dim=1)
        return self.regressor(combined)

In [24]:
# Initialization on M4 Max
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = PlantaGotchiMultiModal().to(device)
print(f"Model initialized on: {device}")

Model initialized on: mps


In [25]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [26]:
model.train() # Set to training mode
running_loss = 0.0

In [27]:
for batch_idx, (images, tabular, labels) in enumerate(train_loader):
    # Move everything to the NPU
    images = images.to(device)
    tabular = tabular.to(device)
    labels = labels.to(device).float() # Labels must be float for MSE

    # --- THE CORE MATH ---
    optimizer.zero_grad()               # Clear old gradients
    outputs = model(images, tabular)    # Forward pass: THE GUESS
    loss = criterion(outputs, labels)   # How wrong were we?
    loss.backward()                     # Backpropagation
    optimizer.step()                    # Update weights
    # ---------------------

    running_loss += loss.item()
    
    if batch_idx % 10 == 0:
        print(f"Batch {batch_idx} | Current Loss: {loss.item():.4f}")

print(f"Epoch finished with Average Loss: {running_loss / len(train_loader):.4f}")

Batch 0 | Current Loss: 6.2543
Batch 10 | Current Loss: 1.1647
Batch 20 | Current Loss: 0.1112
Batch 30 | Current Loss: 0.4388
Batch 40 | Current Loss: 0.2395
Batch 50 | Current Loss: 0.3069
Batch 60 | Current Loss: 0.3836
Batch 70 | Current Loss: 0.1134
Batch 80 | Current Loss: 0.2139
Batch 90 | Current Loss: 0.2742
Batch 100 | Current Loss: 0.1133
Batch 110 | Current Loss: 0.2163
Batch 120 | Current Loss: 0.1335
Batch 130 | Current Loss: 0.1368
Batch 140 | Current Loss: 0.1752
Batch 150 | Current Loss: 0.2053
Batch 160 | Current Loss: 0.3090
Batch 170 | Current Loss: 0.2083
Batch 180 | Current Loss: 0.2123
Batch 190 | Current Loss: 0.1211
Batch 200 | Current Loss: 0.1445
Batch 210 | Current Loss: 0.0842
Batch 220 | Current Loss: 0.1968
Batch 230 | Current Loss: 0.2407
Batch 240 | Current Loss: 0.1285
Batch 250 | Current Loss: 0.2487
Batch 260 | Current Loss: 0.1511
Batch 270 | Current Loss: 0.1772
Batch 280 | Current Loss: 0.2211
Batch 290 | Current Loss: 0.1286
Batch 300 | Current L

In [30]:
PARQUET_VAL_PATH = Path("/Users/mainframe/Workspace/Graduate/notebooks/plant_gotchi_validate.parquet")
PHOTO_VAL_PATH = Path("/Users/mainframe/Workspace/Graduate/dataset/validate/photos")

df_val = pq.read_table(source=PARQUET_VAL_PATH, use_threads=True).to_pandas()

scaler = StandardScaler()
df_val[cols_to_scale] = scaler.fit_transform(df_val[cols_to_scale])

In [31]:
# 1. Initialize the Validation Dataset
val_ds = PlantaDataset(
    dataframe=df_val, 
    img_dir=PHOTO_VAL_PATH, 
    transform=pipeline.transform
)

# 2. Validation Loader (Shuffle is False here; we want to keep it consistent)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=0)

In [32]:
model.eval() # Set to evaluation mode
val_loss = 0.0
total_error = 0.0

with torch.no_grad(): # Disable gradient calculation
    for images, tabular, labels in val_loader:
        # Move to NPU
        images = images.to(device)
        tabular = tabular.to(device)
        labels = labels.to(device).float()

        # Forward pass ONLY
        outputs = model(images, tabular)
        
        # Calculate Loss
        loss = criterion(outputs, labels)
        val_loss += loss.item()
        
        # Calculate MAE (Mean Absolute Error) 
        # This tells you how many 'stages' off you are on average
        total_error += torch.abs(outputs - labels).sum().item()

avg_val_loss = val_loss / len(val_loader)
avg_mae = total_error / len(val_ds)

print(f"Validation Loss: {avg_val_loss:.4f}")
print(f"Mean Absolute Error: {avg_mae:.2f} stages")


Validation Loss: 0.1486
Mean Absolute Error: 0.32 stages


In [33]:
stage_results = {}

for stage in [1.0, 2.0, 3.0]:
    # 1. Filter the dataframe for this specific stage
    stage_df = df_val[df_val['growth'] == stage]
    
    # 2. Initialize a temporary dataset and loader for just this stage
    temp_ds = PlantaDataset(
        dataframe=stage_df,
        img_dir=PHOTO_VAL_PATH,
        transform=pipeline.transform
    )
    temp_loader = DataLoader(temp_ds, batch_size=16, shuffle=False, num_workers=0)
    
    stage_loss = 0.0
    stage_mae_sum = 0.0
    
    model.eval()
    with torch.no_grad():
        for images, tabular, labels in temp_loader: # Using temp_loader here!
            images, tabular, labels = images.to(device), tabular.to(device), labels.to(device).float()
            
            outputs = model(images, tabular)
            loss = criterion(outputs, labels)
            
            stage_loss += loss.item() * images.size(0)
            stage_mae_sum += torch.abs(outputs - labels).sum().item()
    
    # 3. Calculate actual Means
    total_samples = len(temp_ds)
    avg_loss = stage_loss / total_samples
    avg_mae = stage_mae_sum / total_samples
    
    print(f"--- Stage {stage} (n={total_samples}) ---")
    print(f"Average Loss: {avg_loss:.4f}")
    print(f"Mean Absolute Error: {avg_mae:.2f} stages\n")

--- Stage 1.0 (n=124) ---
Average Loss: 0.2455
Mean Absolute Error: 0.47 stages

--- Stage 2.0 (n=1039) ---
Average Loss: 0.0702
Mean Absolute Error: 0.23 stages

--- Stage 3.0 (n=252) ---
Average Loss: 0.4254
Mean Absolute Error: 0.62 stages



In [ ]:
# 시계열 데이터로 변환한 모델 ㄱㄱ
# 위에서 보는건데 데이터 확인해야할 듯...? -> 데이터별 Vision 처리 ㄱㄱ
# Classfier로 변환 하고 Accuracy ㄱㄱ